## Model Training

In [1]:
# Suppress Possible Warnings : In some installations a warning due to pyg_lib path can arise: not relevant to our setting
import warnings
warnings.filterwarnings(
    "ignore",
    message=".*An issue occurred while importing 'pyg-lib'.*",
    category=UserWarning,
    module="torch_geometric.typing"
)
warnings.filterwarnings(
    "ignore",
    message=".*An issue occurred while importing 'torch-sparse'.*",
    category=UserWarning,
    module="torch_geometric.typing"
)

In [2]:
import torch
import logging
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from torchinfo import summary
import torch_geometric.utils as pyg_utils
from torch_geometric import seed_everything

from src.config import set_run_dir
from src.config import set_out_dir
from src.config import load_cfg
from src.logger import setup_printing
from src.utils.device import auto_select_device
from src.model.factory import create_model
from src.train.optim import create_optimizer, create_scheduler
from src.dataset.loader import load_data_and_create_dataloader
from src.train.create_trainer import create_trainer
from src.train.optim import create_criterion
from src.train.task import create_task

%load_ext tensorboard


# Load configuration

In [ ]:
# If you want to change the model configuration, you can change it in the cfg_file
cfg_file = 'run/configs/swat/gdn.yaml'
cfg = load_cfg(cfg_file)
set_out_dir(cfg.out_dir, cfg_file)
set_run_dir(cfg.out_dir, cfg.seed)

setup_printing(logging.INFO)
seed_everything(cfg.seed)
auto_select_device()

print("=========== Model config summary ===========")
logging.info(cfg.model.gdn)
print("=========== Task summary ===========")
logging.info(cfg.task)

# Load Dataset

In [ ]:
loaders, scaler = load_data_and_create_dataloader()

In [ ]:
train_loader, eval_laoder, test_loader = loaders

test_batch = next(iter(test_loader))
test_batch

# Ex 1: inspection the data

## Visualize the test data (pay attention to the batched data)

In [ ]:
# TODO:
x = test_batch.x
print("test_batch.x.shape: ", ...)
n_nodes = cfg.dataset.n_nodes
print("number of features n_nodes: ", ...)
ws =  x.shape[1]
print("window size ws: ", ...)

# get the first graph
g0 = test_batch[0]
print("g0 edge_index: ", ...)

In [ ]:
n_cols = int(np.ceil(np.sqrt(n_nodes)))
n_rows = int(np.ceil(n_nodes / n_cols))
f, axes = plt.subplots(n_rows, n_cols, figsize=(3*n_cols, 2*n_rows))

# TODO: reshape x so that the dimenstion is (batch_size, n_nodes, window_size)
x = x.view(...)
print("x.shape: ", x.shape)

for idx in range(n_nodes):
    axes[idx//n_cols, idx%n_cols].plot(x[:, idx, 0])
    axes[idx//n_cols, idx%n_cols].set_title(f"Node {idx}")

axes[n_nodes//n_cols, n_nodes%n_cols].plot(test_batch.label)
axes[n_nodes//n_cols, n_nodes%n_cols].set_title(f"Attack")

plt.tight_layout()
plt.show()

### Feel free to explore the data

In [ ]:
# TODO: show correlation matrix
corr = np.corrcoef(...)
plt.figure(figsize=(5, 5))
plt.imshow(corr, cmap='viridis')
plt.colorbar()
plt.title("Correlation Matrix")

### Visualize the graph of g.edge_index (we start with a fully connected graph)

In [ ]:
# TODO: use the to_networkx function to convert g0 into a networkx for visulization
g = pyg_utils.to_networkx(...)

# draw the graph
plt.figure(figsize=(5, 5))
nx.draw(g, with_labels=True, font_weight='bold')
plt.show()


# Model

In [ ]:
model = create_model(cfg.model.type)
summary(model)

# Setup training pipeline

In [ ]:
optimizer = create_optimizer(model.parameters())
scheduler = create_scheduler(optimizer)

crit_type = cfg.optim.criterion
criterion = create_criterion(crit_type, mask_loss=False)

task = create_task(
    cfg.task.type,
    task_train_type=cfg.task.train_type,
    track_graph=False,
)

trainer = create_trainer(
    model,
    optimizer=optimizer,
    neptune_writer=None,
    scheduler=scheduler,
    criterion=criterion,
    scaler=scaler,
    task=task,
)

## Training

In [ ]:
# you can interrupt the training by clicking the stop button in the colab UI and the training will be saved
trainer.fit(loaders)

## Using tensorboard

In [ ]:
%tensorboard --logdir run/results/anomaly/swat
